# BSM L05E — Biometria jako *lokalna bramka* do sekretu (SecretBox + policy)

## Tryb pracy
Notebook służy do wypełnienia i wysłania odpowiedzi. Kod implementujesz lokalnie w **Android Studio / Kotlin** w starterze projektu do laboratorium.
## Link do projektu
Wklej link do repozytorium GitHub (to samo repo co w poprzednich labach):

- https://github.com/duszekjk/mobile-systems-security.git


## Przygotowanie przed zajęciami (zrób zanim zaczniesz zadania)

### A. Import projektu w Android Studio (klik po kliku)
1. Uruchom Android Studio.
2. Wybierz: **File → Open...**
3. Wskaż folder projektu w sklonowanym repo, np. `student/apps/lesson_e_app`
4. Jeśli Android Studio zapyta o zaufanie do projektu: wybierz **Trust Project**.
5. Poczekaj na zakończenie synchronizacji Gradle (na dole zobaczysz pasek postępu).

### B. Jeśli Gradle Sync nie startuje lub stoi
1. Kliknij: **File → Sync Project with Gradle Files**
2. Jeśli pojawi się informacja o wymaganej wersji JDK, wybierz JDK 17:
   - **File → Settings → Build, Execution, Deployment → Build Tools → Gradle**
   - **Gradle JDK → 17**

### C. Uruchom testy jednostkowe (bez emulatora)
W tym labie wszystko sprawdzamy przez testy JVM.

Opcja (Android Studio):
1. Otwórz okno Gradle (zwykle po prawej).
2. Rozwiń: `lesson_e_app` → `app` → **Tasks** → **verification**
3. Uruchom: **testDebugUnitTest**

## Cel labu
Wykład 5: biometria zwykle nie jest „sekretem do wysłania”, tylko mechanizmem lokalnym:
- biometria/credential **odblokowuje dostęp** do klucza / sekretu na urządzeniu,
- a dopiero potem aplikacja może wykonać wrażliwą operację.

W tym labie implementujesz:
1) `SecretBox` — minimalne szyfrowanie uwierzytelnione AES-GCM,
2) `BiometricBoundSecretStore` — politykę: sekret można ujawnić tylko z „świeżym” tokenem bramki.

Wszystko weryfikujesz przez testy w `app/src/test/...`.

### D. Jeśli uruchamiasz `./gradlew` z terminala: wymuś JDK 17
Gradle + Kotlin DSL w tym projekcie zakładają Java 17. Jeśli masz domyślne `JAVA_HOME` ustawione na nowszą Javę (np. 25), możesz dostać nieczytelny błąd typu `IllegalArgumentException: 25`.

Sprawdź jaką Javę widzi Gradle:
- `./gradlew -version`

Jeśli nie jest to Java 17, ustaw JDK 17 dla tej sesji terminala (macOS):
- `export JAVA_HOME=$(/usr/libexec/java_home -v 17)`
- `export PATH="$JAVA_HOME/bin:$PATH"`

Potem uruchom ponownie:
- `./gradlew :app:bsmEvidence`

### E. Workflow git (żeby `git pull` nie zjadł Twoich rozwiązań)
Zakładamy, że pracujesz w repozytorium kursu i nie chcesz mieszać rozwiązań między labami.

Przed rozpoczęciem labu (ma być czysto):
- `git status`
- `git checkout main`
- `git pull --rebase`

Na lab: osobny branch:
- `git checkout -b l05e`

W trakcie labu: zapisuj postęp małymi commitami:
- `git add -A && git commit -m "L05E: progress"`

Jeśli musisz dociągnąć zmiany ze startera w trakcie:
- `git checkout main && git pull --rebase`
- `git checkout l05e && git rebase main` (albo: `git merge main`)


In [ ]:
#@title Dane studenta
import requests

Student_ID = "" #@param {type:"string"}
Mail = "" #@param {type:"string"}
Grupa = "" #@param {type:"string"}
Link_do_projektu_Kotlin = "" #@param {type:"string"}

BASE_URL = "https://www.duszekjk.com/bsk/"

def wyslij_odpowiedz(task_id, final_answer):
    final_answer = str(final_answer)
    final_answer_size = len(final_answer)
    final_answer_send = final_answer[:600] + "\n" + str(final_answer_size) + " znaków"
    data = {
        "student_id": Student_ID,
        "student_mail": Mail,
        "task": task_id,
        "grupa": Grupa,
        "answer": final_answer_send,
        "share_link": Link_do_projektu_Kotlin,
    }
    url = BASE_URL + "api/submit_answer/"
    r = requests.post(url, json=data, timeout=20)
    print(r.status_code)
    try:
        j = r.json()
        if "points" in j:
            print("Punkty:", j["points"])
        else:
            print(j)
    except Exception:
        print(r.text)


In [ ]:
answers = {}

def short_text(text, limit=48):
    text = str(text).strip().replace("\n", " ")
    return text[:limit]

def prepare_answer(*parts, limit=220):
    final_answer = "|".join(str(p) for p in parts)
    return final_answer[:limit]

def zapisz_i_wyslij(task_id, final_answer):
    final_answer = str(final_answer)
    answers[task_id] = final_answer
    print(f"Zapisano answers[{task_id}] ({len(final_answer)} znaków)")
    print(final_answer)
    wyslij_odpowiedz(task_id, final_answer)


# E01 — SecretBox: AES-GCM (poufność + integralność)

AES-GCM to szyfrowanie uwierzytelnione (AEAD): jednocześnie zapewnia poufność i integralność.
W praktyce oznacza to, że bez klucza nie da się odczytać danych **ani** sensownie ich podmienić bez wykrycia.
W tym zadaniu budujesz minimalny `SecretBox`, czyli cienką warstwę API ukrywającą detale `Cipher`.
Format `iv || ciphertextAndTag` jest celowo prosty: IV jest jawny i musi iść razem z szyfrogramem.
Najważniejsza zasada bezpieczeństwa GCM: **nigdy nie używaj tego samego IV dwa razy z tym samym kluczem**.

Plik do edycji:
- `student/apps/lesson_e_app/app/src/main/java/com/example/secretlab/secure/SecretBox.kt`

Uzupełnij dokładnie:
- `TODO(L05-1)` w `encrypt(...)`
- `TODO(L05-2)` w `decrypt(...)`

Wymagania:
- algorytm: `AES/GCM/NoPadding`
- IV: 12 bajtów (`SecretBox.IV_BYTES`), a zła długość → `IllegalArgumentException`
- `decrypt(...)` zwraca `null` jeśli:
  - wiadomość jest za krótka
  - autentykacja GCM nie przejdzie (tag/tamper)

## Jak uzyskać odpowiedź do E01 (co wysłać)
1. Uruchom test `SecretBoxStudentTest`.
2. W Terminalu lokalnie uruchom: `./gradlew :app:bsmEvidence`
3. Skopiuj linię `E01|...` do formularza E01 i wyślij.


In [ ]:
#@title E01 — Formularz odpowiedzi
secretbox_tests_passed = ""  #@param ["", "YES", "NO"]

evidence_line = ""  #@param {type:"string"}

final_answer = prepare_answer(
    "pass=" + secretbox_tests_passed,
    "evidence=" + short_text(evidence_line, limit=200),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("E01", final_answer)


# E02 — BiometricBoundSecretStore: polityka „lokalnej bramki”

Wykład 5: biometria (lub PIN urządzenia) działa zwykle jako **lokalna bramka** do klucza/sektu.
To nie jest „sekret do wysłania”, tylko warunek, który musi być spełniony zanim aplikacja ujawni wrażliwe dane.
W labie modelujemy to przez `GateToken` i limit czasu ważności.
Konserwatywna polityka czasu minimalizuje obejścia przez manipulację zegarem.

Plik do edycji:
- `student/apps/lesson_e_app/app/src/main/java/com/example/secretlab/secure/BiometricBoundSecretStore.kt`

Uzupełnij dokładnie:
- `TODO(L05-3)` w `revealSecret(token: GateToken?)`

Polityka:
- `clock()` zwraca **epoch seconds**
- `age = clock() - token.issuedAtEpochSeconds`
- wymaganie: `0 <= age <= maxTokenAgeSeconds`
- jeśli warunki nie są spełnione → zwróć `null`

## Jak uzyskać odpowiedź do E02 (co wysłać)
1. Uruchom test `BiometricBoundSecretStoreStudentTest`.
2. `./gradlew :app:bsmEvidence`
3. Wklej `E02|...` do formularza E02 i wyślij.


In [ ]:
#@title E02 — Formularz odpowiedzi
store_tests_passed = ""  #@param ["", "YES", "NO"]

evidence_line = ""  #@param {type:"string"}

final_answer = prepare_answer(
    "pass=" + store_tests_passed,
    "evidence=" + short_text(evidence_line, limit=200),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("E02", final_answer)


# E03 — AAD: związanie szyfrogramu z kontekstem

AAD (Additional Authenticated Data) pozwala uwierzytelnić dodatkowy kontekst bez szyfrowania go.
To praktyka „purpose binding”: szyfrogram jest ważny tylko w konkretnym celu (np. „seed do TOTP”),
a próba użycia go w innym kontekście ma się nie udać.
W GCM realizuje się to przez `cipher.updateAAD(context)`.

Pliki do edycji:
- `student/apps/lesson_e_app/app/src/main/java/com/example/secretlab/secure/SecretBox.kt`
  - `TODO(L05-5)` w `encryptBound(...)`
  - `TODO(L05-6)` w `decryptBound(...)`
- `student/apps/lesson_e_app/app/src/main/java/com/example/secretlab/secure/BiometricBoundSecretStore.kt`
  - `TODO(L05-4)` w `setSecret(...)`

## Jak uzyskać odpowiedź do E03 (co wysłać)
1. Uruchom `SecretBoxStudentTest` (test `bindsCiphertextToContextWithAad`).
2. `./gradlew :app:bsmEvidence`
3. Wklej `E03|...` do formularza E03 i wyślij.


In [ ]:
#@title E03 — Formularz odpowiedzi
aad_tests_passed = ""  #@param ["", "YES", "NO"]

evidence_line = ""  #@param {type:"string"}

final_answer = prepare_answer(
    "pass=" + aad_tests_passed,
    "evidence=" + short_text(evidence_line, limit=200),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("E03", final_answer)


# E04 — Generowanie IV (nonce): anti-footgun

W E01 IV był parametrem dla deterministycznych testów, ale w prawdziwym kodzie IV musi być generowany poprawnie.
To zadanie dodaje prostą „barierę” na błędy: jedna funkcja `generateIv()` zwraca zawsze 12 bajtów losowych.
To minimalizuje ryzyko, że ktoś:
- użyje stałego IV,
- użyje złej długości,
- lub zacznie ręcznie konstruować IV w wielu miejscach.

Plik do edycji:
- `student/apps/lesson_e_app/app/src/main/java/com/example/secretlab/secure/SecretBox.kt`

Uzupełnij dokładnie:
- `TODO(L05-7)` w `generateIv()`

## Jak uzyskać odpowiedź do E04 (co wysłać)
1. Uruchom `SecretBoxStudentTest` (test `generateIvReturns12BytesAndIsFresh`).
2. `./gradlew :app:bsmEvidence`
3. Wklej `E04|...` do formularza E04 i wyślij.


In [ ]:
#@title E04 — Formularz odpowiedzi
iv_tests_passed = ""  #@param ["", "YES", "NO"]

evidence_line = ""  #@param {type:"string"}

final_answer = prepare_answer(
    "pass=" + iv_tests_passed,
    "evidence=" + short_text(evidence_line, limit=200),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("E04", final_answer)


# E05 — Kontrakt błędów: zawsze `null` (bez wycieków)

W kryptografii aplikacyjnej ważne jest, żeby błąd odszyfrowania nie zdradzał „dlaczego” się nie udało.
Dlatego kontrakt w tym labie jest prosty: decryption zwraca `null` dla każdego niepoprawnego wejścia.
W tym zadaniu sprawdzasz trzy typowe klasy błędu:
- za krótka wiadomość,
- błąd autentykacji (tamper/tag),
- różny kontekst AAD.

Kod do zmiany zwykle nie jest nowy — to raczej dopięcie poprawnych warunków i obsługi wyjątków w E01/E03.

## Jak uzyskać odpowiedź do E05 (co wysłać)
1. Uruchom `SecretBoxStudentTest` (testy: `returnsNullWhenMessageTooShort`, `decryptBoundReturnsNullWhenContextDiffers`, `detectsTamperingAndReturnsNull`).
2. `./gradlew :app:bsmEvidence`
3. Wklej `E05|...` do formularza E05 i wyślij.


In [ ]:
#@title E05 — Formularz odpowiedzi
failure_contract_passed = ""  #@param ["", "YES", "NO"]

evidence_line = ""  #@param {type:"string"}

final_answer = prepare_answer(
    "pass=" + failure_contract_passed,
    "evidence=" + short_text(evidence_line, limit=200),
)
print(final_answer)


In [ ]:
zapisz_i_wyslij("E05", final_answer)
